<a href="https://colab.research.google.com/github/trang1981/ELAPS/blob/main/VIB60ABC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A, B, C dùng đúng cùng 94.453 khách hàng.

Chia cùng một train/test 80:20, random_state=42.

Tạo cùng một 5-fold Stratified CV trên tập train.

Cấu hình mô hình giống nhau: XGBoost, không sampling, missing numeric → 0, categorical → "MISSING", One-Hot.

A = 18 biến; B = 18 biến; C = 19 biến với TENURE_AT_CUTOFF.
Test metrics đầy đủ: AUC, KS, Recall, Precision, F1, Accuracy, Brier, BSS, ECE, MCC, Specificity, NPV, TP/FP/TN/FN.
Ranking Top 10/20/30: Precision, Lift, Capture.
Decile D1–D10.

CV 5 fold đầy đủ tất cả chỉ số.

SHAP cho cả A/B/C, quy về feature gốc.

So sánh A–B, B–C, A–C bằng paired bootstrap 95% CI + paired DeLong.

Không dùng train_test_split() nữa.
Dùng trực tiếp:

FINAL_ABC_MATCHED_E2/exact_E2_shared_train_test_split.csv

FINAL_ABC_MATCHED_E2/exact_E2_shared_CV_folds.csv

Dùng _E2_ORDER để khôi phục đúng thứ tự TRAIN của E2 cũ.

A/B/C dùng đúng cùng 75,562 train + 18,891 test.

A/B/C dùng đúng cùng 5 CV folds.

Cấu hình mô hình đúng E2: XGBoost + None.

Dùng đúng 18 feature của A/B, C thêm TENURE_AT_CUTOFF.

Có kiểm tra A với kết quả E2 cũ.

Có đầy đủ CV, test metrics, ranking, decile, SHAP, bootstrap paired ΔAUC, DeLong và Excel.

In [9]:
# ============================================================
# VIB FLDC-60D
# FINAL HARMONIZED A / B / C
# EXACT E2-LOCKED EXPERIMENT
# ============================================================
#
# GOLDEN E2 FILES:
#
#   FINAL_ABC_MATCHED_E2/
#       exact_E2_shared_train_test_split.csv
#       exact_E2_shared_CV_folds.csv
#
# ============================================================
#
# BRANCH A
#   Fixed 60-Day
#   18 predictors
#
# BRANCH B
#   Event-Anchored
#   18 predictors
#
# BRANCH C
#   Event-Anchored + TENURE_AT_CUTOFF
#   19 predictors
#
# ============================================================
#
# SAME MODEL:
#
#   XGBoost + None
#
#   n_estimators      = 300
#   max_depth         = 6
#   learning_rate     = 0.05
#   subsample         = 0.8
#   colsample_bytree  = 0.8
#   objective         = binary:logistic
#   eval_metric       = logloss
#   random_state      = 42
#
# ============================================================
#
# SAME EXPERIMENT:
#
#   Population = 94,453
#   Train      = 75,562
#   Test       = 18,891
#   CV         = exact E2 folds
#
# ============================================================
#
# OUTPUT:
#
#   1. Exact E2 split
#   2. Exact E2 CV
#   3. A/B/C fold-level CV
#   4. A/B/C CV mean ± SD
#   5. A/B/C test metrics
#   6. Confusion matrix metrics
#   7. Ranking 10/20/30%
#   8. Decile table
#   9. Test predictions
#   10. SHAP global importance
#   11. A vs E2 reference
#   12. Pairwise Delta AUC
#   13. Paired bootstrap 95% CI
#   14. Paired DeLong
#   15. Excel workbook
#
# GOOGLE COLAB — COPY AND RUN DIRECTLY
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

!pip -q install xgboost scipy shap openpyxl


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import warnings

import numpy as np
import pandas as pd

from pathlib import Path
from scipy.stats import norm

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    recall_score,
    precision_score,
    f1_score,
    accuracy_score,
    brier_score_loss,
    confusion_matrix,
    matthews_corrcoef
)

from xgboost import XGBClassifier

import shap

from google.colab import drive

warnings.filterwarnings("ignore")


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

print("=" * 120)
print("MOUNT GOOGLE DRIVE")
print("=" * 120)

drive.mount(
    "/content/drive",
    force_remount=False
)

ROOT = Path(
    "/content/drive/MyDrive/Fintect"
)

if not ROOT.is_dir():
    raise RuntimeError(
        "Không truy cập được Google Drive:\n"
        f"{ROOT}"
    )


# ============================================================
# 3. GLOBAL CONFIGURATION
# ============================================================

SEED = 42

N_SPLITS = 5

DECISION_THRESHOLD = 0.50

SHAP_SAMPLE_SIZE = 5000

N_BOOTSTRAP = 5000

EXPECTED_CUSTOMERS = 94453
EXPECTED_TRAIN = 75562
EXPECTED_TEST = 18891


# ============================================================
# 4. GOLDEN E2 REFERENCE
# ============================================================

E2_REFERENCE = {
    "CV_AUC_MEAN": 0.909226,
    "CV_AUC_SD": 0.005908,
    "TEST_AUC": 0.910242,
    "TEST_KS": 0.673291,
    "TEST_RECALL": 0.168000,
    "PRECISION_AT_10": 0.296825,
    "LIFT_AT_10": 6.408376,
    "CAPTURE_AT_10": 0.641143
}

REFERENCE_TOLERANCE = 0.002


# ============================================================
# 5. FILE CONFIGURATION
# ============================================================

A_FILE = (
    ROOT
    / "final_landmark_60d"
    / "final_dataset_landmark_60d_no_auto_job.csv"
)

B_FILE = (
    ROOT
    / "VIB_FLDC_60D.csv"
)

C_FILE = (
    ROOT
    / "VIB_E1_BRANCH_C.csv"
)

E2_SPLIT_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_train_test_split.csv"
)

E2_CV_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_CV_folds.csv"
)

OUTPUT_DIR = (
    ROOT
    / "FINAL_ABC_EXACT_E2_FIXED"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 6. ID / TARGET
# ============================================================

ID_COL = "CUSTOMER_NUMBER"

TARGET_A = "TARGET_60D"
TARGET_B = "COUNT_CREDITCARD"
TARGET_C = "COUNT_CREDITCARD"

TENURE_COL = "TENURE_AT_CUTOFF"


# ============================================================
# 7. EXACT FEATURE SET
# ============================================================

BASE_FEATURES = [
    "CLIENT_SEX",
    "EB_REGISTER_CHANNEL",
    "SMS",
    "VERIFY_METHOD",
    "AGE",
    "LOGIN_PER_ACTIVE_DAY",
    "INTEREST_RATE_RATIO",
    "TRANS_LV1_MODE",
    "TRANS_LV2_MODE",
    "TRANS_AMOUNT_MAX",
    "TRANS_AMOUNT_MEAN",
    "COUNT_CA_ACCT",
    "AVG_CA_BALANCE",
    "COUNT_TD_ACCT",
    "AVG_TD_BALANCE",
    "COUNT_OF_LOAN",
    "AVG_LOAN_AMOUNT",
    "TOTAL_LOAN_AMOUNT"
]

FEATURES_A = list(BASE_FEATURES)
FEATURES_B = list(BASE_FEATURES)
FEATURES_C = BASE_FEATURES + [TENURE_COL]

if len(FEATURES_A) != 18:
    raise RuntimeError("A phải có 18 features.")

if len(FEATURES_B) != 18:
    raise RuntimeError("B phải có 18 features.")

if len(FEATURES_C) != 19:
    raise RuntimeError("C phải có 19 features.")

if FEATURES_A != FEATURES_B:
    raise RuntimeError("Feature set A và B không giống nhau.")


# ============================================================
# 8. XGBOOST CONFIGURATION
# ============================================================

XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "logloss",
    "objective": "binary:logistic",
    "random_state": SEED,
    "n_jobs": -1
}


def get_xgb():
    return XGBClassifier(
        **XGB_PARAMS
    )


# ============================================================
# 9. STANDARDIZE CUSTOMER ID
# ============================================================

def standardize_customer_id(series):

    result = (
        series
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    invalid_values = [
        "",
        "NAN",
        "<NA>",
        "NONE",
        "NULL"
    ]

    result = result.mask(
        result.str.upper().isin(
            invalid_values
        ),
        pd.NA
    )

    return result


# ============================================================
# 10. LOAD BRANCH
# ============================================================

def load_branch(
    file_path,
    target_col,
    branch_name
):

    print("\n" + "-" * 110)
    print(f"LOAD {branch_name}")
    print(file_path)

    if not Path(file_path).is_file():
        raise FileNotFoundError(
            f"Không tìm thấy:\n{file_path}"
        )

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.upper()
    )

    required = [
        ID_COL,
        target_col
    ]

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{branch_name} thiếu cột:\n{missing}"
        )

    df[ID_COL] = standardize_customer_id(
        df[ID_COL]
    )

    df["TARGET_SHARED"] = pd.to_numeric(
        df[target_col],
        errors="coerce"
    )

    if TENURE_COL in df.columns:
        df[TENURE_COL] = pd.to_numeric(
            df[TENURE_COL],
            errors="coerce"
        )

    df = (
        df
        .dropna(
            subset=[
                ID_COL,
                "TARGET_SHARED"
            ]
        )
        .copy()
    )

    df["TARGET_SHARED"] = (
        df["TARGET_SHARED"]
        .astype(int)
    )

    if not df["TARGET_SHARED"].isin(
        [0, 1]
    ).all():
        raise ValueError(
            f"{branch_name}: TARGET không phải 0/1."
        )

    duplicate_count = int(
        df[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{branch_name}: "
            f"{duplicate_count:,} duplicate customers."
        )

    print("Shape:", df.shape)
    print(
        "Customers:",
        f"{df[ID_COL].nunique():,}"
    )
    print(
        "Positive:",
        int(df["TARGET_SHARED"].sum())
    )

    return df


# ============================================================
# 11. LOAD A / B / C
# ============================================================

print("\n" + "=" * 120)
print("LOAD A / B / C")
print("=" * 120)

df_a = load_branch(
    A_FILE,
    TARGET_A,
    "A"
)

df_b = load_branch(
    B_FILE,
    TARGET_B,
    "B"
)

df_c = load_branch(
    C_FILE,
    TARGET_C,
    "C"
)


# ============================================================
# 12. LOAD EXACT E2 TRAIN / TEST SPLIT
# ============================================================

print("\n" + "=" * 120)
print("LOAD EXACT E2 TRAIN / TEST SPLIT")
print("=" * 120)

if not E2_SPLIT_FILE.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_SPLIT_FILE}"
    )

e2_split = pd.read_csv(
    E2_SPLIT_FILE,
    low_memory=False
)

e2_split.columns = (
    e2_split.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_split_columns = [
    ID_COL,
    "TARGET",
    "SPLIT"
]

missing = [
    c
    for c in required_split_columns
    if c not in e2_split.columns
]

if missing:
    raise ValueError(
        "E2 split thiếu cột:\n"
        f"{missing}"
    )

e2_split[ID_COL] = (
    standardize_customer_id(
        e2_split[ID_COL]
    )
)

e2_split["TARGET"] = pd.to_numeric(
    e2_split["TARGET"],
    errors="coerce"
)

e2_split["SPLIT"] = (
    e2_split["SPLIT"]
    .astype("string")
    .str.strip()
    .str.upper()
)

e2_split = (
    e2_split
    .dropna(
        subset=[
            ID_COL,
            "TARGET"
        ]
    )
    .copy()
)

e2_split["TARGET"] = (
    e2_split["TARGET"]
    .astype(int)
)


# ============================================================
# 13. LOAD EXACT E2 CV
# ============================================================

print("\n" + "=" * 120)
print("LOAD EXACT E2 CV")
print("=" * 120)

if not E2_CV_FILE.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_CV_FILE}"
    )

e2_cv = pd.read_csv(
    E2_CV_FILE,
    low_memory=False
)

e2_cv.columns = (
    e2_cv.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_cv_columns = [
    ID_COL,
    "TARGET",
    "_E2_ORDER",
    "CV_FOLD"
]

missing = [
    c
    for c in required_cv_columns
    if c not in e2_cv.columns
]

if missing:
    raise ValueError(
        "E2 CV thiếu cột:\n"
        f"{missing}"
    )

e2_cv[ID_COL] = (
    standardize_customer_id(
        e2_cv[ID_COL]
    )
)

e2_cv["TARGET"] = pd.to_numeric(
    e2_cv["TARGET"],
    errors="coerce"
)

e2_cv["_E2_ORDER"] = pd.to_numeric(
    e2_cv["_E2_ORDER"],
    errors="coerce"
)

e2_cv["CV_FOLD"] = pd.to_numeric(
    e2_cv["CV_FOLD"],
    errors="coerce"
)

e2_cv = (
    e2_cv
    .dropna(
        subset=[
            ID_COL,
            "TARGET",
            "_E2_ORDER",
            "CV_FOLD"
        ]
    )
    .copy()
)

e2_cv["TARGET"] = (
    e2_cv["TARGET"]
    .astype(int)
)

e2_cv["_E2_ORDER"] = (
    e2_cv["_E2_ORDER"]
    .astype(int)
)

e2_cv["CV_FOLD"] = (
    e2_cv["CV_FOLD"]
    .astype(int)
)


# ============================================================
# 14. POPULATION CHECK
# ============================================================

ids_a = set(df_a[ID_COL])
ids_b = set(df_b[ID_COL])
ids_c = set(df_c[ID_COL])
ids_split = set(e2_split[ID_COL])
ids_cv = set(e2_cv[ID_COL])

print("\n" + "=" * 120)
print("POPULATION CHECK")
print("=" * 120)

print("A:", f"{len(ids_a):,}")
print("B:", f"{len(ids_b):,}")
print("C:", f"{len(ids_c):,}")
print("E2 split:", f"{len(ids_split):,}")
print("E2 CV:", f"{len(ids_cv):,}")

if len(ids_a) != EXPECTED_CUSTOMERS:
    raise RuntimeError(
        "A không có đúng 94,453 customers."
    )

if ids_a != ids_b:
    raise RuntimeError(
        "B population khác A."
    )

if ids_a != ids_c:
    raise RuntimeError(
        "C population khác A."
    )

if ids_a != ids_split:
    raise RuntimeError(
        "E2 split population khác A."
    )


# ============================================================
# 15. TARGET CONSISTENCY
# ============================================================

target_a = (
    df_a[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_A"
        }
    )
)

target_b = (
    df_b[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_B"
        }
    )
)

target_c = (
    df_c[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_C"
        }
    )
)

target_check = (
    target_a
    .merge(
        target_b,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        target_c,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        e2_split[
            [ID_COL, "TARGET"]
        ],
        on=ID_COL,
        validate="one_to_one"
    )
)

target_mismatch = (
    (target_check["TARGET_A"] != target_check["TARGET_B"])
    |
    (target_check["TARGET_A"] != target_check["TARGET_C"])
    |
    (target_check["TARGET_A"] != target_check["TARGET"])
)

print(
    "Target mismatch:",
    int(target_mismatch.sum())
)

if target_mismatch.any():
    raise RuntimeError(
        "Target A/B/C/E2 không giống nhau."
    )


# ============================================================
# 16. EXACT E2 TRAIN / TEST
# ============================================================

train_ids = set(
    e2_split.loc[
        e2_split["SPLIT"] == "TRAIN",
        ID_COL
    ]
)

test_ids = set(
    e2_split.loc[
        e2_split["SPLIT"] == "TEST",
        ID_COL
    ]
)

print("\n" + "=" * 120)
print("EXACT E2 TRAIN / TEST")
print("=" * 120)

print(
    "Train:",
    f"{len(train_ids):,}"
)

print(
    "Test:",
    f"{len(test_ids):,}"
)

print(
    "Train base rate:",
    f"{e2_split.loc[e2_split['SPLIT']=='TRAIN','TARGET'].mean():.4%}"
)

print(
    "Test base rate:",
    f"{e2_split.loc[e2_split['SPLIT']=='TEST','TARGET'].mean():.4%}"
)

if len(train_ids) != EXPECTED_TRAIN:
    raise RuntimeError(
        "TRAIN không phải 75,562."
    )

if len(test_ids) != EXPECTED_TEST:
    raise RuntimeError(
        "TEST không phải 18,891."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "TRAIN / TEST overlap."
    )

if (train_ids | test_ids) != ids_a:
    raise RuntimeError(
        "TRAIN + TEST không bao phủ population A."
    )


# ============================================================
# 17. EXACT E2 CV ORDER
# ============================================================

e2_cv = (
    e2_cv
    .sort_values("_E2_ORDER")
    .reset_index(drop=True)
)

expected_order = np.arange(
    len(e2_cv)
)

if not np.array_equal(
    e2_cv["_E2_ORDER"].to_numpy(),
    expected_order
):
    raise RuntimeError(
        "_E2_ORDER không liên tục từ 0 đến N-1."
    )

if len(e2_cv) != EXPECTED_TRAIN:
    raise RuntimeError(
        "E2 CV không có đúng 75,562 records."
    )

if set(e2_cv[ID_COL]) != train_ids:
    raise RuntimeError(
        "E2 CV customers khác E2 TRAIN."
    )

if e2_cv["CV_FOLD"].nunique() != N_SPLITS:
    raise RuntimeError(
        "E2 CV không có đủ 5 folds."
    )

print("\n" + "=" * 120)
print("EXACT E2 CV")
print("=" * 120)

print(
    e2_cv[
        "CV_FOLD"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 18. GOLDEN ORDER
# ============================================================

golden_train_order = (
    e2_cv[
        ID_COL
    ]
    .astype(str)
    .tolist()
)

golden_test_order = (
    e2_split.loc[
        e2_split["SPLIT"] == "TEST",
        ID_COL
    ]
    .astype(str)
    .tolist()
)


# ============================================================
# 19. PREPROCESSOR
# ============================================================

def build_preprocessor(
    X_train
):

    numeric_cols = (
        X_train
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = (
        X_train
        .select_dtypes(
            exclude=np.number
        )
        .columns
        .tolist()
    )

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        )
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipe,
            numeric_cols
        ),
        (
            "cat",
            categorical_pipe,
            categorical_cols
        )
    ])

    return preprocessor


# ============================================================
# 20. METRICS
# ============================================================

def ks_score(
    y_true,
    prob
):

    fpr, tpr, _ = roc_curve(
        y_true,
        prob
    )

    return float(
        np.max(
            tpr - fpr
        )
    )


def bss_score(
    y_true,
    prob
):

    brier = brier_score_loss(
        y_true,
        prob
    )

    prevalence = float(
        np.mean(y_true)
    )

    reference = (
        prevalence *
        (1 - prevalence)
    )

    if reference <= 0:
        return np.nan

    return float(
        1 -
        brier / reference
    )


def ece_score(
    y_true,
    prob,
    n_bins=10
):

    y_true = np.asarray(y_true)
    prob = np.asarray(prob)

    bins = np.linspace(
        0,
        1,
        n_bins + 1
    )

    ece = 0.0

    for i in range(n_bins):

        if i == n_bins - 1:

            mask = (
                (prob >= bins[i])
                &
                (prob <= bins[i + 1])
            )

        else:

            mask = (
                (prob >= bins[i])
                &
                (prob < bins[i + 1])
            )

        if not mask.any():
            continue

        observed = (
            y_true[mask].mean()
        )

        predicted = (
            prob[mask].mean()
        )

        weight = (
            mask.sum() /
            len(y_true)
        )

        ece += (
            weight *
            abs(
                observed -
                predicted
            )
        )

    return float(ece)


def ranking_metrics(
    y_true,
    prob
):

    temp = pd.DataFrame({
        "Y_TRUE": np.asarray(y_true),
        "PROB": np.asarray(prob)
    })

    temp = (
        temp
        .sort_values(
            "PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    base_rate = float(
        temp["Y_TRUE"].mean()
    )

    total_positive = float(
        temp["Y_TRUE"].sum()
    )

    rows = []

    for rate in [
        0.10,
        0.20,
        0.30
    ]:

        top_n = max(
            1,
            int(
                np.ceil(
                    len(temp) * rate
                )
            )
        )

        top_df = (
            temp
            .iloc[:top_n]
        )

        precision = float(
            top_df["Y_TRUE"].mean()
        )

        lift = (
            precision / base_rate
            if base_rate > 0
            else np.nan
        )

        capture = (
            float(
                top_df["Y_TRUE"].sum()
            ) /
            total_positive
            if total_positive > 0
            else np.nan
        )

        rows.append({
            "TOP_RATE": rate,
            "TOP_K": f"Top {int(rate * 100)}%",
            "CUSTOMERS": top_n,
            "PRECISION": precision,
            "LIFT": float(lift),
            "CAPTURE": float(capture)
        })

    return pd.DataFrame(rows)


def decile_metrics(
    y_true,
    prob
):

    temp = pd.DataFrame({
        "Y_TRUE": np.asarray(y_true),
        "PROB": np.asarray(prob)
    })

    temp = (
        temp
        .sort_values(
            "PROB",
            ascending=False
        )
        .reset_index(drop=True)
    )

    temp["DECILE"] = pd.qcut(
        np.arange(len(temp)),
        q=10,
        labels=[
            f"D{i}"
            for i in range(1, 11)
        ]
    )

    base_rate = (
        temp["Y_TRUE"].mean()
    )

    result = (
        temp
        .groupby(
            "DECILE",
            observed=False
        )
        .agg(
            CUSTOMERS=(
                "Y_TRUE",
                "count"
            ),
            POSITIVES=(
                "Y_TRUE",
                "sum"
            ),
            OBSERVED_RATE=(
                "Y_TRUE",
                "mean"
            ),
            MEAN_PROBABILITY=(
                "PROB",
                "mean"
            )
        )
        .reset_index()
    )

    result["LIFT"] = (
        result["OBSERVED_RATE"] /
        base_rate
    )

    return result


# ============================================================
# 21. SHAP HELPERS
# ============================================================

def map_transformed_features(
    transformed_names,
    original_features
):

    sorted_features = sorted(
        original_features,
        key=len,
        reverse=True
    )

    rows = []

    for transformed_name in transformed_names:

        clean_name = transformed_name

        if clean_name.startswith("num__"):
            clean_name = clean_name[5:]

        elif clean_name.startswith("cat__"):
            clean_name = clean_name[5:]

        matched = None

        for feature in sorted_features:

            if clean_name == feature:
                matched = feature
                break

            if clean_name.startswith(
                feature + "_"
            ):
                matched = feature
                break

        if matched is None:
            matched = clean_name

        rows.append({
            "TRANSFORMED_FEATURE":
                transformed_name,
            "ORIGINAL_FEATURE":
                matched
        })

    return pd.DataFrame(rows)


def aggregate_shap(
    shap_values,
    mapping,
    original_features
):

    labels = (
        mapping["ORIGINAL_FEATURE"]
        .tolist()
    )

    rows = []

    for feature in original_features:

        indices = [
            i
            for i, label in enumerate(labels)
            if label == feature
        ]

        if not indices:

            rows.append({
                "FEATURE": feature,
                "MEAN_ABS_SHAP": 0.0,
                "MEAN_SIGNED_SHAP": 0.0
            })

            continue

        selected = (
            shap_values[:, indices]
        )

        signed = (
            selected.sum(axis=1)
        )

        absolute = (
            np.abs(selected).sum(axis=1)
        )

        rows.append({
            "FEATURE": feature,
            "MEAN_ABS_SHAP": float(
                absolute.mean()
            ),
            "MEAN_SIGNED_SHAP": float(
                signed.mean()
            )
        })

    result = pd.DataFrame(rows)

    result = (
        result
        .sort_values(
            "MEAN_ABS_SHAP",
            ascending=False
        )
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "RANK",
        np.arange(
            1,
            len(result) + 1
        )
    )

    return result


# ============================================================
# 22. RUN ONE BRANCH
# ============================================================

def run_branch(
    branch_name,
    data,
    feature_cols
):

    print(
        "\n" + "=" * 120
    )
    print(
        f"RUN {branch_name}"
    )
    print(
        "=" * 120
    )

    # --------------------------------------------------------
    # Feature check
    # --------------------------------------------------------

    missing_features = [
        feature
        for feature in feature_cols
        if feature not in data.columns
    ]

    if missing_features:
        raise ValueError(
            f"{branch_name} thiếu features:\n"
            +
            "\n".join(missing_features)
        )

    branch = data.copy()

    branch[ID_COL] = (
        standardize_customer_id(
            branch[ID_COL]
        )
    )

    branch_indexed = (
        branch
        .set_index(
            ID_COL,
            drop=False
        )
    )

    # --------------------------------------------------------
    # Check indexed population
    # --------------------------------------------------------

    if branch_indexed.index.duplicated().any():
        raise RuntimeError(
            f"{branch_name}: duplicate CUSTOMER_NUMBER."
        )

    # --------------------------------------------------------
    # Exact E2 TRAIN order
    # --------------------------------------------------------

    missing_train = (
        set(golden_train_order)
        -
        set(branch_indexed.index)
    )

    if missing_train:
        raise RuntimeError(
            f"{branch_name}: "
            f"thiếu {len(missing_train)} "
            "customers trong TRAIN."
        )

    train_part = (
        branch_indexed
        .loc[golden_train_order]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Exact E2 TEST population
    # --------------------------------------------------------

    missing_test = (
        set(golden_test_order)
        -
        set(branch_indexed.index)
    )

    if missing_test:
        raise RuntimeError(
            f"{branch_name}: "
            f"thiếu {len(missing_test)} "
            "customers trong TEST."
        )

    test_part = (
        branch_indexed
        .loc[golden_test_order]
        .copy()
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # Exact TRAIN order check
    # --------------------------------------------------------

    actual_train_ids = (
        train_part[ID_COL]
        .astype(str)
        .to_numpy()
    )

    expected_train_ids = np.asarray(
        golden_train_order,
        dtype=str
    )

    if not np.array_equal(
        actual_train_ids,
        expected_train_ids
    ):

        mismatch_positions = np.where(
            actual_train_ids !=
            expected_train_ids
        )[0]

        print(
            "\nTRAIN ORDER MISMATCH"
        )

        print(
            "Mismatch count:",
            len(mismatch_positions)
        )

        for position in (
            mismatch_positions[:20]
        ):

            print(
                f"Position {position}: "
                f"actual={actual_train_ids[position]} | "
                f"expected={expected_train_ids[position]}"
            )

        raise RuntimeError(
            f"{branch_name}: "
            "TRAIN order không khớp _E2_ORDER."
        )

    print(
        f"{branch_name}: "
        "EXACT E2 TRAIN ORDER VERIFIED."
    )

    # --------------------------------------------------------
    # Check against E2 CV file
    # --------------------------------------------------------

    e2_cv_ids = (
        e2_cv[ID_COL]
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        actual_train_ids,
        e2_cv_ids
    ):
        raise RuntimeError(
            f"{branch_name}: "
            "TRAIN order không khớp E2 CV."
        )

    # --------------------------------------------------------
    # X / y
    # --------------------------------------------------------

    X_train = (
        train_part[feature_cols]
        .reset_index(drop=True)
    )

    X_test = (
        test_part[feature_cols]
        .reset_index(drop=True)
    )

    y_train = (
        train_part["TARGET_SHARED"]
        .astype(int)
        .reset_index(drop=True)
    )

    y_test = (
        test_part["TARGET_SHARED"]
        .astype(int)
        .reset_index(drop=True)
    )

    train_customer_ids = (
        train_part[ID_COL]
        .astype(str)
        .reset_index(drop=True)
    )

    test_customer_ids = (
        test_part[ID_COL]
        .astype(str)
        .reset_index(drop=True)
    )

    print(
        "Train:",
        f"{len(X_train):,}"
    )

    print(
        "Test:",
        f"{len(X_test):,}"
    )

    print(
        "Features:",
        len(feature_cols)
    )

    # --------------------------------------------------------
    # Exact E2 CV assignment
    # --------------------------------------------------------

    fold_assignment = (
        e2_cv["CV_FOLD"]
        .to_numpy(dtype=int)
    )

    if len(fold_assignment) != len(X_train):
        raise RuntimeError(
            f"{branch_name}: CV length mismatch."
        )

    # ========================================================
    # FIVE-FOLD CV
    # ========================================================

    cv_rows = []

    for fold in range(
        1,
        N_SPLITS + 1
    ):

        valid_mask = (
            fold_assignment == fold
        )

        train_mask_fold = (
            ~valid_mask
        )

        X_fold_train = (
            X_train.loc[
                train_mask_fold
            ]
            .copy()
        )

        X_fold_valid = (
            X_train.loc[
                valid_mask
            ]
            .copy()
        )

        y_fold_train = (
            y_train.loc[
                train_mask_fold
            ]
            .to_numpy()
        )

        y_fold_valid = (
            y_train.loc[
                valid_mask
            ]
            .to_numpy()
        )

        # ----------------------------------------------------
        # Preprocess
        # ----------------------------------------------------

        preprocessor = build_preprocessor(
            X_fold_train
        )

        X_fold_train_processed = (
            preprocessor
            .fit_transform(
                X_fold_train
            )
        )

        X_fold_valid_processed = (
            preprocessor
            .transform(
                X_fold_valid
            )
        )

        # ----------------------------------------------------
        # XGBoost + None
        # ----------------------------------------------------

        model = get_xgb()

        model.fit(
            X_fold_train_processed,
            y_fold_train
        )

        fold_prob = (
            model
            .predict_proba(
                X_fold_valid_processed
            )[:, 1]
        )

        fold_pred = (
            fold_prob >=
            DECISION_THRESHOLD
        ).astype(int)

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        fold_auc = roc_auc_score(
            y_fold_valid,
            fold_prob
        )

        fold_ks = ks_score(
            y_fold_valid,
            fold_prob
        )

        fold_recall = recall_score(
            y_fold_valid,
            fold_pred,
            zero_division=0
        )

        fold_precision = precision_score(
            y_fold_valid,
            fold_pred,
            zero_division=0
        )

        fold_f1 = f1_score(
            y_fold_valid,
            fold_pred,
            zero_division=0
        )

        fold_accuracy = accuracy_score(
            y_fold_valid,
            fold_pred
        )

        fold_brier = brier_score_loss(
            y_fold_valid,
            fold_prob
        )

        fold_bss = bss_score(
            y_fold_valid,
            fold_prob
        )

        fold_ece = ece_score(
            y_fold_valid,
            fold_prob
        )

        cv_rows.append({
            "FOLD":
                fold,

            "N_VALID":
                len(y_fold_valid),

            "AUC":
                float(fold_auc),

            "KS":
                float(fold_ks),

            "Recall":
                float(fold_recall),

            "Precision":
                float(fold_precision),

            "F1":
                float(fold_f1),

            "Accuracy":
                float(fold_accuracy),

            "Brier":
                float(fold_brier),

            "BSS":
                float(fold_bss),

            "ECE":
                float(fold_ece)
        })

        print(
            f"Fold {fold}: "
            f"AUC={fold_auc:.4f} | "
            f"KS={fold_ks:.4f} | "
            f"Recall={fold_recall:.4f} | "
            f"Precision={fold_precision:.4f} | "
            f"F1={fold_f1:.4f} | "
            f"Brier={fold_brier:.4f}"
        )

        del model
        del preprocessor
        del X_fold_train_processed
        del X_fold_valid_processed

        gc.collect()

    cv_results = pd.DataFrame(
        cv_rows
    )

    # ========================================================
    # FINAL MODEL
    # ========================================================

    final_preprocessor = build_preprocessor(
        X_train
    )

    X_train_processed = (
        final_preprocessor
        .fit_transform(
            X_train
        )
    )

    X_test_processed = (
        final_preprocessor
        .transform(
            X_test
        )
    )

    final_model = get_xgb()

    final_model.fit(
        X_train_processed,
        y_train.to_numpy()
    )

    test_prob = (
        final_model
        .predict_proba(
            X_test_processed
        )[:, 1]
    )

    test_pred = (
        test_prob >=
        DECISION_THRESHOLD
    ).astype(int)

    # ========================================================
    # TEST METRICS
    # ========================================================

    test_auc = roc_auc_score(
        y_test,
        test_prob
    )

    test_ks = ks_score(
        y_test,
        test_prob
    )

    test_recall = recall_score(
        y_test,
        test_pred,
        zero_division=0
    )

    test_precision = precision_score(
        y_test,
        test_pred,
        zero_division=0
    )

    test_f1 = f1_score(
        y_test,
        test_pred,
        zero_division=0
    )

    test_accuracy = accuracy_score(
        y_test,
        test_pred
    )

    test_brier = brier_score_loss(
        y_test,
        test_prob
    )

    test_bss = bss_score(
        y_test,
        test_prob
    )

    test_ece = ece_score(
        y_test,
        test_prob
    )

    test_mcc = matthews_corrcoef(
        y_test,
        test_pred
    )

    tn, fp, fn, tp = (
        confusion_matrix(
            y_test,
            test_pred,
            labels=[0, 1]
        )
        .ravel()
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    npv = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    # ========================================================
    # RANKING / DECILES
    # ========================================================

    ranking = ranking_metrics(
        y_test,
        test_prob
    )

    deciles = decile_metrics(
        y_test,
        test_prob
    )

    # ========================================================
    # PREDICTIONS
    # ========================================================

    predictions = pd.DataFrame({
        ID_COL:
            test_customer_ids.to_numpy(),

        "Y_TRUE":
            y_test.to_numpy(),

        "Y_PROB":
            test_prob,

        "Y_PRED":
            test_pred
    })

    # ========================================================
    # SHAP
    # ========================================================

    shap_mapping = pd.DataFrame()

    try:

        shap_n = min(
            SHAP_SAMPLE_SIZE,
            len(X_test_processed)
        )

        rng = np.random.default_rng(
            SEED
        )

        shap_indices = rng.choice(
            len(X_test_processed),
            size=shap_n,
            replace=False
        )

        X_shap = (
            X_test_processed[
                shap_indices
            ]
        )

        explainer = shap.TreeExplainer(
            final_model
        )

        shap_raw = (
            explainer
            .shap_values(
                X_shap
            )
        )

        if isinstance(
            shap_raw,
            list
        ):
            shap_values = np.asarray(
                shap_raw[-1]
            )
        else:
            shap_values = np.asarray(
                shap_raw
            )

        if shap_values.ndim == 3:
            shap_values = (
                shap_values[:, :, -1]
            )

        transformed_names = (
            final_preprocessor
            .get_feature_names_out()
            .tolist()
        )

        shap_mapping = (
            map_transformed_features(
                transformed_names,
                feature_cols
            )
        )

        shap_result = (
            aggregate_shap(
                shap_values,
                shap_mapping,
                feature_cols
            )
        )

        top_shap = str(
            shap_result.iloc[0]["FEATURE"]
        )

    except Exception as e:

        print(
            f"\nSHAP warning for {branch_name}:",
            repr(e)
        )

        shap_result = pd.DataFrame({
            "RANK":
                np.arange(
                    1,
                    len(feature_cols) + 1
                ),

            "FEATURE":
                feature_cols,

            "MEAN_ABS_SHAP":
                np.nan,

            "MEAN_SIGNED_SHAP":
                np.nan
        })

        top_shap = None

    # ========================================================
    # CV SUMMARY
    # ========================================================

    cv_summary_rows = []

    for metric in [
        "AUC",
        "KS",
        "Recall",
        "Precision",
        "F1",
        "Accuracy",
        "Brier",
        "BSS",
        "ECE"
    ]:

        mean_value = float(
            cv_results[metric].mean()
        )

        sd_value = float(
            cv_results[metric].std(
                ddof=1
            )
        )

        cv_summary_rows.append({
            "METRIC":
                metric,

            "MEAN":
                mean_value,

            "SD":
                sd_value,

            "MEAN_PLUS_MINUS_SD":
                (
                    f"{mean_value:.4f} ± "
                    f"{sd_value:.4f}"
                )
        })

    cv_summary = pd.DataFrame(
        cv_summary_rows
    )

    # ========================================================
    # FULL METRICS
    # ========================================================

    metrics = {
        "Branch":
            branch_name,

        "Features":
            len(feature_cols),

        "N_Customers":
            len(data),

        "N_Train":
            len(X_train),

        "N_Test":
            len(X_test),

        "Train_Base_Rate":
            float(y_train.mean()),

        "Test_Base_Rate":
            float(y_test.mean()),

        # CV
        "CV_AUC_Mean":
            float(
                cv_results["AUC"].mean()
            ),

        "CV_AUC_SD":
            float(
                cv_results["AUC"].std(
                    ddof=1
                )
            ),

        "CV_KS_Mean":
            float(
                cv_results["KS"].mean()
            ),

        "CV_KS_SD":
            float(
                cv_results["KS"].std(
                    ddof=1
                )
            ),

        "CV_Recall_Mean":
            float(
                cv_results["Recall"].mean()
            ),

        "CV_Recall_SD":
            float(
                cv_results["Recall"].std(
                    ddof=1
                )
            ),

        "CV_Precision_Mean":
            float(
                cv_results["Precision"].mean()
            ),

        "CV_Precision_SD":
            float(
                cv_results["Precision"].std(
                    ddof=1
                )
            ),

        "CV_F1_Mean":
            float(
                cv_results["F1"].mean()
            ),

        "CV_F1_SD":
            float(
                cv_results["F1"].std(
                    ddof=1
                )
            ),

        "CV_Accuracy_Mean":
            float(
                cv_results["Accuracy"].mean()
            ),

        "CV_Accuracy_SD":
            float(
                cv_results["Accuracy"].std(
                    ddof=1
                )
            ),

        "CV_Brier_Mean":
            float(
                cv_results["Brier"].mean()
            ),

        "CV_Brier_SD":
            float(
                cv_results["Brier"].std(
                    ddof=1
                )
            ),

        "CV_BSS_Mean":
            float(
                cv_results["BSS"].mean()
            ),

        "CV_BSS_SD":
            float(
                cv_results["BSS"].std(
                    ddof=1
                )
            ),

        "CV_ECE_Mean":
            float(
                cv_results["ECE"].mean()
            ),

        "CV_ECE_SD":
            float(
                cv_results["ECE"].std(
                    ddof=1
                )
            ),

        # TEST
        "Test_AUC":
            float(test_auc),

        "Test_KS":
            float(test_ks),

        "Test_Recall":
            float(test_recall),

        "Test_Precision":
            float(test_precision),

        "Test_F1":
            float(test_f1),

        "Test_Accuracy":
            float(test_accuracy),

        "Test_Brier":
            float(test_brier),

        "Test_BSS":
            float(test_bss),

        "Test_ECE":
            float(test_ece),

        "Test_MCC":
            float(test_mcc),

        "Test_Specificity":
            float(specificity),

        "Test_NPV":
            float(npv),

        "TP":
            int(tp),

        "FP":
            int(fp),

        "TN":
            int(tn),

        "FN":
            int(fn),

        "Top_SHAP_Feature":
            top_shap
    }

    for _, row in ranking.iterrows():

        pct = int(
            row["TOP_RATE"] * 100
        )

        metrics[
            f"Precision@{pct}%"
        ] = float(
            row["PRECISION"]
        )

        metrics[
            f"Lift@{pct}%"
        ] = float(
            row["LIFT"]
        )

        metrics[
            f"Capture@{pct}%"
        ] = float(
            row["CAPTURE"]
        )

    # ========================================================
    # PRINT
    # ========================================================

    print("\nFINAL TEST RESULT")

    print(
        f"AUC         = {test_auc:.6f}"
    )

    print(
        f"KS          = {test_ks:.6f}"
    )

    print(
        f"Recall      = {test_recall:.6f}"
    )

    print(
        f"Precision   = {test_precision:.6f}"
    )

    print(
        f"F1          = {test_f1:.6f}"
    )

    print(
        f"Accuracy    = {test_accuracy:.6f}"
    )

    print(
        f"Brier       = {test_brier:.6f}"
    )

    print(
        f"BSS         = {test_bss:.6f}"
    )

    print(
        f"ECE         = {test_ece:.6f}"
    )

    print(
        f"MCC         = {test_mcc:.6f}"
    )

    print(
        f"Specificity = {specificity:.6f}"
    )

    print(
        f"NPV         = {npv:.6f}"
    )

    print(
        f"Precision@10 = "
        f"{metrics['Precision@10%']:.6f}"
    )

    print(
        f"Lift@10      = "
        f"{metrics['Lift@10%']:.6f}"
    )

    print(
        f"Capture@10   = "
        f"{metrics['Capture@10%']:.6f}"
    )

    print(
        f"Precision@20 = "
        f"{metrics['Precision@20%']:.6f}"
    )

    print(
        f"Lift@20      = "
        f"{metrics['Lift@20%']:.6f}"
    )

    print(
        f"Capture@20   = "
        f"{metrics['Capture@20%']:.6f}"
    )

    print(
        f"Precision@30 = "
        f"{metrics['Precision@30%']:.6f}"
    )

    print(
        f"Lift@30      = "
        f"{metrics['Lift@30%']:.6f}"
    )

    print(
        f"Capture@30   = "
        f"{metrics['Capture@30%']:.6f}"
    )

    print(
        "Top SHAP:",
        top_shap
    )

    return {
        "metrics":
            metrics,

        "cv":
            cv_results,

        "cv_summary":
            cv_summary,

        "ranking":
            ranking,

        "deciles":
            deciles,

        "predictions":
            predictions,

        "shap":
            shap_result,

        "shap_mapping":
            shap_mapping,

        "model":
            final_model,

        "preprocessor":
            final_preprocessor
    }


# ============================================================
# 23. RUN BRANCH A
# ============================================================

result_a = run_branch(
    "A — Fixed 60-Day",
    df_a,
    FEATURES_A
)

gc.collect()


# ============================================================
# 24. A VS GOLDEN E2
# ============================================================

a_metrics = result_a["metrics"]

reference_rows = [
    {
        "Metric": "CV AUC",
        "E2": E2_REFERENCE["CV_AUC_MEAN"],
        "Current A": a_metrics["CV_AUC_Mean"]
    },
    {
        "Metric": "CV AUC SD",
        "E2": E2_REFERENCE["CV_AUC_SD"],
        "Current A": a_metrics["CV_AUC_SD"]
    },
    {
        "Metric": "Test AUC",
        "E2": E2_REFERENCE["TEST_AUC"],
        "Current A": a_metrics["Test_AUC"]
    },
    {
        "Metric": "KS",
        "E2": E2_REFERENCE["TEST_KS"],
        "Current A": a_metrics["Test_KS"]
    },
    {
        "Metric": "Recall",
        "E2": E2_REFERENCE["TEST_RECALL"],
        "Current A": a_metrics["Test_Recall"]
    },
    {
        "Metric": "Precision@10",
        "E2": E2_REFERENCE["PRECISION_AT_10"],
        "Current A": a_metrics["Precision@10%"]
    },
    {
        "Metric": "Lift@10",
        "E2": E2_REFERENCE["LIFT_AT_10"],
        "Current A": a_metrics["Lift@10%"]
    },
    {
        "Metric": "Capture@10",
        "E2": E2_REFERENCE["CAPTURE_AT_10"],
        "Current A": a_metrics["Capture@10%"]
    }
]

comparison_reference = pd.DataFrame(
    reference_rows
)

comparison_reference[
    "Absolute_Difference"
] = (
    comparison_reference["Current A"]
    -
    comparison_reference["E2"]
).abs()

print(
    "\n" + "=" * 120
)

print(
    "A VS GOLDEN E2 REFERENCE"
)

print(
    "=" * 120
)

print(
    comparison_reference.to_string(
        index=False
    )
)

auc_diff = abs(
    a_metrics["Test_AUC"]
    -
    E2_REFERENCE["TEST_AUC"]
)

ks_diff = abs(
    a_metrics["Test_KS"]
    -
    E2_REFERENCE["TEST_KS"]
)

print(
    "\nTest AUC difference:",
    f"{auc_diff:.8f}"
)

print(
    "KS difference:",
    f"{ks_diff:.8f}"
)

if (
    auc_diff <= REFERENCE_TOLERANCE
    and
    ks_diff <= REFERENCE_TOLERANCE
):

    print(
        "\nA is within the E2 reference tolerance."
    )

else:

    print(
        "\nWARNING: A chưa khớp hoàn toàn số E2 cũ."
    )

    print(
        "Population, split và CV hiện đã được khóa."
    )

    print(
        "Không dừng B/C để giữ được bộ so sánh A/B/C."
    )


# ============================================================
# 25. RUN BRANCH B
# ============================================================

result_b = run_branch(
    "B — Event-Anchored",
    df_b,
    FEATURES_B
)

gc.collect()


# ============================================================
# 26. RUN BRANCH C
# ============================================================

result_c = run_branch(
    "C — Event-Anchored + Tenure",
    df_c,
    FEATURES_C
)

gc.collect()


# ============================================================
# 27. TEST PREDICTIONS
# ============================================================

pred_a = (
    result_a["predictions"]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

pred_b = (
    result_b["predictions"]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)

pred_c = (
    result_c["predictions"]
    .sort_values(ID_COL)
    .reset_index(drop=True)
)


# ============================================================
# 28. SAME TEST CUSTOMERS
# ============================================================

test_set_a = set(pred_a[ID_COL])
test_set_b = set(pred_b[ID_COL])
test_set_c = set(pred_c[ID_COL])

print(
    "\n" + "=" * 120
)

print(
    "TEST CUSTOMER VERIFICATION"
)

print(
    "=" * 120
)

print(
    "A == B:",
    test_set_a == test_set_b
)

print(
    "B == C:",
    test_set_b == test_set_c
)

print(
    "A == C:",
    test_set_a == test_set_c
)

if not (
    test_set_a ==
    test_set_b ==
    test_set_c
):
    raise RuntimeError(
        "A/B/C không cùng TEST customer set."
    )


# ============================================================
# 29. SAME TEST LABELS
# ============================================================

label_check = (
    pred_a[
        [ID_COL, "Y_TRUE"]
    ]
    .merge(
        pred_b[
            [ID_COL, "Y_TRUE"]
        ],
        on=ID_COL,
        suffixes=(
            "_A",
            "_B"
        ),
        validate="one_to_one"
    )
    .merge(
        pred_c[
            [ID_COL, "Y_TRUE"]
        ],
        on=ID_COL,
        validate="one_to_one"
    )
)

if (
    (
        label_check["Y_TRUE_A"]
        !=
        label_check["Y_TRUE_B"]
    )
    |
    (
        label_check["Y_TRUE_A"]
        !=
        label_check["Y_TRUE"]
    )
).any():

    raise RuntimeError(
        "TARGET test A/B/C không giống nhau."
    )


# ============================================================
# 30. ALIGN PREDICTIONS FOR PAIRED COMPARISON
# ============================================================

common_test = (
    pred_a[
        [ID_COL, "Y_TRUE"]
    ]
    .copy()
)

aligned_a = (
    common_test
    .merge(
        pred_a[
            [ID_COL, "Y_PROB"]
        ],
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )
)

aligned_b = (
    common_test
    .merge(
        pred_b[
            [ID_COL, "Y_PROB"]
        ],
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )
)

aligned_c = (
    common_test
    .merge(
        pred_c[
            [ID_COL, "Y_PROB"]
        ],
        on=ID_COL,
        how="left",
        validate="one_to_one"
    )
)

y_test_common = (
    common_test[
        "Y_TRUE"
    ]
    .to_numpy(dtype=int)
)

score_a = (
    aligned_a[
        "Y_PROB"
    ]
    .to_numpy(dtype=float)
)

score_b = (
    aligned_b[
        "Y_PROB"
    ]
    .to_numpy(dtype=float)
)

score_c = (
    aligned_c[
        "Y_PROB"
    ]
    .to_numpy(dtype=float)
)


# ============================================================
# 31. PAIRED BOOTSTRAP
# ============================================================

def paired_bootstrap_delta_auc(
    y,
    score_1,
    score_2,
    n_bootstrap=5000,
    seed=42
):

    y = np.asarray(y)
    score_1 = np.asarray(score_1)
    score_2 = np.asarray(score_2)

    n = len(y)

    rng = np.random.default_rng(
        seed
    )

    deltas = []

    for _ in range(
        n_bootstrap
    ):

        indices = rng.integers(
            0,
            n,
            size=n
        )

        y_boot = (
            y[indices]
        )

        if np.unique(
            y_boot
        ).size < 2:
            continue

        auc_1 = roc_auc_score(
            y_boot,
            score_1[indices]
        )

        auc_2 = roc_auc_score(
            y_boot,
            score_2[indices]
        )

        deltas.append(
            auc_2 - auc_1
        )

    deltas = np.asarray(
        deltas
    )

    ci_low, ci_high = np.percentile(
        deltas,
        [2.5, 97.5]
    )

    return (
        float(ci_low),
        float(ci_high),
        deltas
    )


# ============================================================
# 32. DELONG
# ============================================================

def compute_midrank(x):

    x = np.asarray(
        x,
        dtype=float
    )

    order = np.argsort(x)

    sorted_x = x[order]

    midranks_sorted = np.empty(
        len(x),
        dtype=float
    )

    start = 0

    while start < len(x):

        end = start

        while (
            end + 1 < len(x)
            and
            sorted_x[end + 1]
            ==
            sorted_x[start]
        ):

            end += 1

        midranks_sorted[
            start:end + 1
        ] = (
            start +
            end +
            2
        ) / 2.0

        start = end + 1

    midranks = np.empty(
        len(x),
        dtype=float
    )

    midranks[order] = (
        midranks_sorted
    )

    return midranks


def fast_delong(
    predictions_sorted_transposed,
    label_1_count
):

    m = int(label_1_count)

    n = (
        predictions_sorted_transposed.shape[1]
        -
        m
    )

    k = (
        predictions_sorted_transposed.shape[0]
    )

    tx = np.empty(
        (k, m)
    )

    ty = np.empty(
        (k, n)
    )

    tz = np.empty(
        (k, m + n)
    )

    for r in range(k):

        prediction = (
            predictions_sorted_transposed[r]
        )

        tx[r] = compute_midrank(
            prediction[:m]
        )

        ty[r] = compute_midrank(
            prediction[m:]
        )

        tz[r] = compute_midrank(
            prediction
        )

    aucs = (
        tz[:, :m].sum(axis=1)
        /
        m
        /
        n
        -
        (m + 1.0) /
        (2.0 * n)
    )

    v01 = (
        tz[:, :m] / n
        -
        tx / n
    )

    v10 = (
        1.0
        -
        (
            tz[:, m:]
            -
            ty
        )
        / m
    )

    sx = np.cov(v01)
    sy = np.cov(v10)

    covariance = (
        sx / m
        +
        sy / n
    )

    return (
        aucs,
        covariance
    )


def paired_delong(
    y,
    score_1,
    score_2
):

    order = np.argsort(-y)

    y_sorted = y[order]

    predictions = np.vstack([
        score_1[order],
        score_2[order]
    ])

    m = int(
        y_sorted.sum()
    )

    aucs, covariance = fast_delong(
        predictions,
        m
    )

    contrast = np.array([
        -1.0,
        1.0
    ])

    delta = float(
        contrast @ aucs
    )

    variance = float(
        contrast
        @ covariance
        @ contrast
    )

    if variance <= 0:
        raise RuntimeError(
            "DeLong variance <= 0."
        )

    se = float(
        np.sqrt(variance)
    )

    z = float(
        delta / se
    )

    p_value = float(
        2 *
        norm.sf(
            abs(z)
        )
    )

    return (
        delta,
        se,
        z,
        p_value
    )


# ============================================================
# 33. PAIRWISE COMPARISON
# ============================================================

def compare_pair(
    name_1,
    name_2,
    score_1,
    score_2
):

    auc_1 = roc_auc_score(
        y_test_common,
        score_1
    )

    auc_2 = roc_auc_score(
        y_test_common,
        score_2
    )

    delta = (
        auc_2 -
        auc_1
    )

    (
        ci_low,
        ci_high,
        bootstrap_values
    ) = paired_bootstrap_delta_auc(
        y_test_common,
        score_1,
        score_2,
        n_bootstrap=N_BOOTSTRAP,
        seed=SEED
    )

    (
        delong_delta,
        delong_se,
        delong_z,
        delong_p
    ) = paired_delong(
        y_test_common,
        score_1,
        score_2
    )

    return (
        {
            "Comparison":
                f"{name_2} - {name_1}",

            "AUC_1":
                float(auc_1),

            "AUC_2":
                float(auc_2),

            "Delta_AUC":
                float(delta),

            "Bootstrap_95CI_Lower":
                float(ci_low),

            "Bootstrap_95CI_Upper":
                float(ci_high),

            "DeLong_SE":
                float(delong_se),

            "DeLong_Delta":
                float(delong_delta),

            "DeLong_Z":
                float(delong_z),

            "DeLong_p":
                float(delong_p),

            "Bootstrap_N":
                len(bootstrap_values)
        },
        bootstrap_values
    )


ab_result, bootstrap_ab = compare_pair(
    "A",
    "B",
    score_a,
    score_b
)

bc_result, bootstrap_bc = compare_pair(
    "B",
    "C",
    score_b,
    score_c
)

ac_result, bootstrap_ac = compare_pair(
    "A",
    "C",
    score_a,
    score_c
)

pairwise_table = pd.DataFrame([
    ab_result,
    bc_result,
    ac_result
])


# ============================================================
# 34. FULL METRICS TABLE
# ============================================================

full_metrics = pd.DataFrame([
    result_a["metrics"],
    result_b["metrics"],
    result_c["metrics"]
])


# ============================================================
# 35. PUBLICATION TABLE
# ============================================================

publication_columns = [
    "Branch",
    "Features",
    "N_Customers",
    "N_Train",
    "N_Test",

    "Train_Base_Rate",
    "Test_Base_Rate",

    "CV_AUC_Mean",
    "CV_AUC_SD",

    "CV_KS_Mean",
    "CV_KS_SD",

    "CV_Recall_Mean",
    "CV_Recall_SD",

    "CV_Precision_Mean",
    "CV_Precision_SD",

    "CV_F1_Mean",
    "CV_F1_SD",

    "CV_Accuracy_Mean",
    "CV_Accuracy_SD",

    "CV_Brier_Mean",
    "CV_Brier_SD",

    "CV_BSS_Mean",
    "CV_BSS_SD",

    "CV_ECE_Mean",
    "CV_ECE_SD",

    "Test_AUC",
    "Test_KS",
    "Test_Recall",
    "Test_Precision",
    "Test_F1",
    "Test_Accuracy",
    "Test_Brier",
    "Test_BSS",
    "Test_ECE",
    "Test_MCC",
    "Test_Specificity",
    "Test_NPV",

    "Precision@10%",
    "Lift@10%",
    "Capture@10%",

    "Precision@20%",
    "Lift@20%",
    "Capture@20%",

    "Precision@30%",
    "Lift@30%",
    "Capture@30%",

    "TP",
    "FP",
    "TN",
    "FN",

    "Top_SHAP_Feature"
]

publication_table = (
    full_metrics[
        publication_columns
    ]
    .copy()
)


# ============================================================
# 36. SAVE POPULATION / SPLIT / CV
# ============================================================

population_table = (
    target_check[
        [
            ID_COL,
            "TARGET_A"
        ]
    ]
    .rename(
        columns={
            "TARGET_A":
                "TARGET"
        }
    )
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

population_table.to_csv(
    OUTPUT_DIR /
    "ABC_common_population_94453.csv",
    index=False,
    encoding="utf-8-sig"
)

e2_split.to_csv(
    OUTPUT_DIR /
    "EXACT_E2_shared_train_test_split_used.csv",
    index=False,
    encoding="utf-8-sig"
)

e2_cv.to_csv(
    OUTPUT_DIR /
    "EXACT_E2_shared_CV_folds_used.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 37. SAVE TEST PREDICTIONS
# ============================================================

pred_a.to_csv(
    OUTPUT_DIR /
    "Branch_A_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

pred_b.to_csv(
    OUTPUT_DIR /
    "Branch_B_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

pred_c.to_csv(
    OUTPUT_DIR /
    "Branch_C_test_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 38. SAVE CV / RANKING / DECILES / SHAP
# ============================================================

for branch_code, result in [
    ("A", result_a),
    ("B", result_b),
    ("C", result_c)
]:

    result["cv"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_CV.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["cv_summary"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_CV_summary.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["ranking"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_ranking.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["deciles"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_deciles.csv",
        index=False,
        encoding="utf-8-sig"
    )

    result["shap"].to_csv(
        OUTPUT_DIR /
        f"Branch_{branch_code}_SHAP.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 39. SAVE MAIN METRICS
# ============================================================

full_metrics.to_csv(
    OUTPUT_DIR /
    "ABC_full_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

publication_table.to_csv(
    OUTPUT_DIR /
    "ABC_publication_table.csv",
    index=False,
    encoding="utf-8-sig"
)

pairwise_table.to_csv(
    OUTPUT_DIR /
    "ABC_pairwise_DeltaAUC.csv",
    index=False,
    encoding="utf-8-sig"
)

comparison_reference.to_csv(
    OUTPUT_DIR /
    "A_vs_E2_reference.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 40. SAVE BOOTSTRAP DISTRIBUTIONS
# ============================================================

pd.DataFrame({
    "Delta_AUC_B_minus_A":
        bootstrap_ab
}).to_csv(
    OUTPUT_DIR /
    "Bootstrap_B_minus_A.csv",
    index=False
)

pd.DataFrame({
    "Delta_AUC_C_minus_B":
        bootstrap_bc
}).to_csv(
    OUTPUT_DIR /
    "Bootstrap_C_minus_B.csv",
    index=False
)

pd.DataFrame({
    "Delta_AUC_C_minus_A":
        bootstrap_ac
}).to_csv(
    OUTPUT_DIR /
    "Bootstrap_C_minus_A.csv",
    index=False
)


# ============================================================
# 41. EXCEL WORKBOOK
# ============================================================

excel_file = (
    OUTPUT_DIR /
    "ABC_FINAL_EXACT_E2_RESULTS.xlsx"
)

with pd.ExcelWriter(
    excel_file,
    engine="openpyxl"
) as writer:

    population_table.to_excel(
        writer,
        sheet_name="Population",
        index=False
    )

    e2_split.to_excel(
        writer,
        sheet_name="Exact E2 Split",
        index=False
    )

    e2_cv.to_excel(
        writer,
        sheet_name="Exact E2 CV",
        index=False
    )

    comparison_reference.to_excel(
        writer,
        sheet_name="A vs E2",
        index=False
    )

    publication_table.to_excel(
        writer,
        sheet_name="Publication Table",
        index=False
    )

    full_metrics.to_excel(
        writer,
        sheet_name="Full Metrics",
        index=False
    )

    pairwise_table.to_excel(
        writer,
        sheet_name="Pairwise DeltaAUC",
        index=False
    )

    for branch_code, result in [
        ("A", result_a),
        ("B", result_b),
        ("C", result_c)
    ]:

        result["cv"].to_excel(
            writer,
            sheet_name=f"{branch_code} CV",
            index=False
        )

        result["cv_summary"].to_excel(
            writer,
            sheet_name=f"{branch_code} CV Summary",
            index=False
        )

        result["ranking"].to_excel(
            writer,
            sheet_name=f"{branch_code} Ranking",
            index=False
        )

        result["deciles"].to_excel(
            writer,
            sheet_name=f"{branch_code} Deciles",
            index=False
        )

        result["predictions"].to_excel(
            writer,
            sheet_name=f"{branch_code} Predictions",
            index=False
        )

        result["shap"].to_excel(
            writer,
            sheet_name=f"{branch_code} SHAP",
            index=False
        )


# ============================================================
# 42. FINAL VALIDATION
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "FINAL VALIDATION"
)

print(
    "=" * 150
)

# Population
if len(ids_a) != EXPECTED_CUSTOMERS:
    raise RuntimeError(
        "Population A không đúng."
    )

if len(train_ids) != EXPECTED_TRAIN:
    raise RuntimeError(
        "TRAIN population không đúng."
    )

if len(test_ids) != EXPECTED_TEST:
    raise RuntimeError(
        "TEST population không đúng."
    )

# Same test customers
if test_set_a != test_set_b:
    raise RuntimeError(
        "TEST A != TEST B."
    )

if test_set_b != test_set_c:
    raise RuntimeError(
        "TEST B != TEST C."
    )

# Same test labels
y_true_a = (
    pred_a["Y_TRUE"]
    .to_numpy(dtype=int)
)

y_true_b = (
    pred_b["Y_TRUE"]
    .to_numpy(dtype=int)
)

y_true_c = (
    pred_c["Y_TRUE"]
    .to_numpy(dtype=int)
)

if not np.array_equal(
    y_true_a,
    y_true_b
):
    raise RuntimeError(
        "Y_TRUE A != B."
    )

if not np.array_equal(
    y_true_a,
    y_true_c
):
    raise RuntimeError(
        "Y_TRUE A != C."
    )

# Test duplicate check
for code, prediction in [
    ("A", pred_a),
    ("B", pred_b),
    ("C", pred_c)
]:

    duplicate_count = int(
        prediction[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise RuntimeError(
            f"Branch {code} có "
            f"{duplicate_count} "
            "duplicate test customers."
        )

# Exact E2 CV length
if len(e2_cv) != EXPECTED_TRAIN:
    raise RuntimeError(
        "E2 CV length không đúng."
    )

# Exact E2 order
if not np.array_equal(
    e2_cv["_E2_ORDER"].to_numpy(),
    np.arange(
        EXPECTED_TRAIN
    )
):
    raise RuntimeError(
        "E2_ORDER không đúng."
    )

print(
    "Population check       : PASS"
)

print(
    "TRAIN population       : PASS"
)

print(
    "TEST population        : PASS"
)

print(
    "A/B/C TEST customers   : PASS"
)

print(
    "A/B/C TEST labels      : PASS"
)

print(
    "No duplicate TEST IDs  : PASS"
)

print(
    "Exact E2 TRAIN order   : PASS"
)

print(
    "Exact E2 CV            : PASS"
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "FINAL A / B / C — EXACT E2"
)

print(
    "=" * 150
)

print(
    "\nPopulation:",
    f"{EXPECTED_CUSTOMERS:,}"
)

print(
    "Train:",
    f"{EXPECTED_TRAIN:,}"
)

print(
    "Test:",
    f"{EXPECTED_TEST:,}"
)

print(
    "\nA TEST AUC:",
    f"{result_a['metrics']['Test_AUC']:.6f}"
)

print(
    "B TEST AUC:",
    f"{result_b['metrics']['Test_AUC']:.6f}"
)

print(
    "C TEST AUC:",
    f"{result_c['metrics']['Test_AUC']:.6f}"
)

print(
    "\nA TEST KS:",
    f"{result_a['metrics']['Test_KS']:.6f}"
)

print(
    "B TEST KS:",
    f"{result_b['metrics']['Test_KS']:.6f}"
)

print(
    "C TEST KS:",
    f"{result_c['metrics']['Test_KS']:.6f}"
)

print(
    "\n" +
    "A/B/C same TEST customers:",
    test_set_a == test_set_b == test_set_c
)


# ============================================================
# 44. PAIRWISE RESULTS
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "PAIRWISE DELTA AUC"
)

print(
    "=" * 150
)

print(
    pairwise_table[
        [
            "Comparison",
            "AUC_1",
            "AUC_2",
            "Delta_AUC",
            "Bootstrap_95CI_Lower",
            "Bootstrap_95CI_Upper",
            "DeLong_Delta",
            "DeLong_Z",
            "DeLong_p"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 45. PUBLICATION SUMMARY
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "PUBLICATION SUMMARY"
)

print(
    "=" * 150
)

print(
    publication_table.to_string(
        index=False
    )
)


# ============================================================
# 46. A VS E2 FINAL
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "A VS GOLDEN E2 REFERENCE"
)

print(
    "=" * 150
)

print(
    comparison_reference.to_string(
        index=False
    )
)

print(
    "\nTest AUC absolute difference:",
    f"{auc_diff:.8f}"
)

print(
    "KS absolute difference:",
    f"{ks_diff:.8f}"
)


# ============================================================
# 47. OUTPUT
# ============================================================

print(
    "\n" + "=" * 150
)

print(
    "COMPLETED"
)

print(
    "=" * 150
)

print(
    "\nOutput directory:"
)

print(
    OUTPUT_DIR
)

print(
    "\nExcel:"
)

print(
    excel_file
)

print(
    "\nExact E2 split used:"
)

print(
    E2_SPLIT_FILE
)

print(
    "\nExact E2 CV used:"
)

print(
    E2_CV_FILE
)

print(
    "\nALL FINAL VALIDATION CHECKS PASSED."
)

MOUNT GOOGLE DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

LOAD A / B / C

--------------------------------------------------------------------------------------------------------------
LOAD A
/content/drive/MyDrive/Fintect/final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv
Shape: (94453, 21)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD B
/content/drive/MyDrive/Fintect/VIB_FLDC_60D.csv
Shape: (94453, 25)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD C
/content/drive/MyDrive/Fintect/VIB_E1_BRANCH_C.csv
Shape: (94453, 28)
Customers: 94,453
Positive: 4374

LOAD EXACT E2 TRAIN / TEST SPLIT

LOAD EXACT E2 CV

POPULATION CHECK
A: 94,453
B: 94,453
C: 94,453
E2 split: 94,453
E2 CV: 75,562


In [10]:
# ============================================================
# TABLE 10 — SAME-POPULATION DECOMPOSITION
#
# Branch A : Fixed first 60 days       — 18 predictors
# Branch B : Event anchored            — 18 predictors
# Branch C : Event anchored + tenure   — 19 predictors
#
# CONTROLLED EXPERIMENT
# ------------------------------------------------------------
# FIXED ACROSS A / B / C:
#   - Same 94,453 customers
#   - Same target
#   - Same train/test customers
#   - Same 5-fold CV folds
#   - Same XGBoost configuration
#   - No sampling
#   - Same preprocessing
#
# ONLY DIFFERENCE:
#   A/B/C feature construction
#
# OUTPUT:
#   - Table 10
#   - A/B/C detailed CV
#   - A/B/C test predictions
#   - Pairwise Delta AUC
#   - Excel workbook
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

!pip -q install xgboost openpyxl


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import warnings

import numpy as np
import pandas as pd

from pathlib import Path
from google.colab import drive

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    brier_score_loss
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

print("=" * 110)
print("MOUNT GOOGLE DRIVE")
print("=" * 110)

drive.mount(
    "/content/drive",
    force_remount=False
)

MYDRIVE_ROOT = "/content/drive/MyDrive"

if not Path(MYDRIVE_ROOT).is_dir():
    raise RuntimeError(
        f"Không truy cập được Google Drive:\n{MYDRIVE_ROOT}"
    )


# ============================================================
# 3. CONFIGURATION
# ============================================================

ROOT = Path(
    MYDRIVE_ROOT
) / "Fintect"

SEED = 42

TEST_SIZE = 0.20
N_SPLITS = 5

DECISION_THRESHOLD = 0.50


# ============================================================
# 4. INPUT FILES
# ============================================================

BRANCH_A_FILE = (
    ROOT
    / "final_landmark_60d"
    / "final_dataset_landmark_60d_no_auto_job.csv"
)

BRANCH_B_FILE = (
    ROOT
    / "VIB_FLDC_60D.csv"
)

BRANCH_C_FILE = (
    ROOT
    / "VIB_E1_BRANCH_C.csv"
)


# ============================================================
# 5. GOLDEN E2 SPLIT / CV
# ============================================================
#
# Đây là các file đã được kiểm tra:
#
# exact_E2_shared_train_test_split.csv
# exact_E2_shared_CV_folds.csv
#
# Không tạo split mới.
# ============================================================

E2_SPLIT_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_train_test_split.csv"
)

E2_CV_FILE = (
    ROOT
    / "FINAL_ABC_MATCHED_E2"
    / "exact_E2_shared_CV_folds.csv"
)


# ============================================================
# 6. OUTPUT
# ============================================================

OUTPUT_DIR = (
    ROOT
    / "TABLE_10_SAME_POPULATION"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE10_FILE = (
    OUTPUT_DIR
    / "Table_10_same_population_decomposition.csv"
)

CV_A_FILE = (
    OUTPUT_DIR
    / "Branch_A_CV.csv"
)

CV_B_FILE = (
    OUTPUT_DIR
    / "Branch_B_CV.csv"
)

CV_C_FILE = (
    OUTPUT_DIR
    / "Branch_C_CV.csv"
)

PRED_A_FILE = (
    OUTPUT_DIR
    / "Branch_A_test_predictions.csv"
)

PRED_B_FILE = (
    OUTPUT_DIR
    / "Branch_B_test_predictions.csv"
)

PRED_C_FILE = (
    OUTPUT_DIR
    / "Branch_C_test_predictions.csv"
)

PAIRWISE_FILE = (
    OUTPUT_DIR
    / "Table_10_pairwise_delta_auc.csv"
)

EXCEL_FILE = (
    OUTPUT_DIR
    / "Table_10_same_population_decomposition.xlsx"
)


# ============================================================
# 7. COLUMN CONFIGURATION
# ============================================================

ID_COL = "CUSTOMER_NUMBER"

TENURE_COL = "TENURE_AT_CUTOFF"


BASE_FEATURES = [

    "CLIENT_SEX",

    "EB_REGISTER_CHANNEL",

    "SMS",

    "VERIFY_METHOD",

    "AGE",

    "LOGIN_PER_ACTIVE_DAY",

    "INTEREST_RATE_RATIO",

    "TRANS_LV1_MODE",

    "TRANS_LV2_MODE",

    "TRANS_AMOUNT_MAX",

    "TRANS_AMOUNT_MEAN",

    "COUNT_CA_ACCT",

    "AVG_CA_BALANCE",

    "COUNT_TD_ACCT",

    "AVG_TD_BALANCE",

    "COUNT_OF_LOAN",

    "AVG_LOAN_AMOUNT",

    "TOTAL_LOAN_AMOUNT"

]

FEATURES_A = list(
    BASE_FEATURES
)

FEATURES_B = list(
    BASE_FEATURES
)

FEATURES_C = (
    BASE_FEATURES
    + [TENURE_COL]
)

assert len(BASE_FEATURES) == 18
assert len(FEATURES_A) == 18
assert len(FEATURES_B) == 18
assert len(FEATURES_C) == 19


# ============================================================
# 8. XGBOOST
# ============================================================
#
# Phải dùng đúng cấu hình E2 / controlled A-B-C.
#
# ============================================================

XGB_PARAMS = {

    "n_estimators": 300,

    "max_depth": 6,

    "learning_rate": 0.05,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "objective": "binary:logistic",

    "eval_metric": "logloss",

    "tree_method": "hist",

    "max_bin": 256,

    "random_state": SEED,

    "n_jobs": 2
}


def get_xgb():

    return XGBClassifier(
        **XGB_PARAMS
    )


# ============================================================
# 9. STANDARDIZE ID
# ============================================================

def standardize_customer_id(series):

    result = (
        series
        .astype("string")
        .str.strip()
        .str.replace(
            r"\.0$",
            "",
            regex=True
        )
    )

    invalid_values = [
        "",
        "NAN",
        "<NA>",
        "NONE",
        "NULL"
    ]

    result = result.mask(
        result.str.upper().isin(
            invalid_values
        ),
        pd.NA
    )

    return result


# ============================================================
# 10. LOAD BRANCH
# ============================================================

def load_branch(
    file_path,
    branch_name,
    target_col
):

    print("\n" + "-" * 110)
    print(f"LOAD {branch_name}")
    print(file_path)
    print("-" * 110)

    if not Path(file_path).is_file():
        raise FileNotFoundError(
            f"Không tìm thấy file:\n{file_path}"
        )

    df = pd.read_csv(
        file_path,
        low_memory=False
    )

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.upper()
    )

    required = [
        ID_COL,
        target_col
    ]

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{branch_name} thiếu cột:\n{missing}"
        )

    df[ID_COL] = standardize_customer_id(
        df[ID_COL]
    )

    df["TARGET_SHARED"] = pd.to_numeric(
        df[target_col],
        errors="coerce"
    )

    df = df.dropna(
        subset=[
            ID_COL,
            "TARGET_SHARED"
        ]
    ).copy()

    df["TARGET_SHARED"] = (
        df["TARGET_SHARED"]
        .astype(int)
    )

    if not df["TARGET_SHARED"].isin(
        [0, 1]
    ).all():

        raise ValueError(
            f"{branch_name}: target không phải 0/1."
        )

    duplicate_count = int(
        df[ID_COL]
        .duplicated()
        .sum()
    )

    if duplicate_count > 0:
        raise ValueError(
            f"{branch_name}: "
            f"{duplicate_count:,} duplicate customers."
        )

    print(
        "Shape:",
        df.shape
    )

    print(
        "Customers:",
        f"{df[ID_COL].nunique():,}"
    )

    print(
        "Positive:",
        int(df["TARGET_SHARED"].sum())
    )

    return df


# ============================================================
# 11. LOAD A / B / C
# ============================================================

print("\n" + "=" * 110)
print("LOAD A / B / C")
print("=" * 110)

df_a = load_branch(
    BRANCH_A_FILE,
    "A",
    "TARGET_60D"
)

df_b = load_branch(
    BRANCH_B_FILE,
    "B",
    "COUNT_CREDITCARD"
)

df_c = load_branch(
    BRANCH_C_FILE,
    "C",
    "COUNT_CREDITCARD"
)


# ============================================================
# 12. LOAD GOLDEN E2 SPLIT
# ============================================================

print("\n" + "=" * 110)
print("LOAD GOLDEN E2 TRAIN / TEST SPLIT")
print("=" * 110)

if not E2_SPLIT_FILE.is_file():

    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_SPLIT_FILE}"
    )

split_df = pd.read_csv(
    E2_SPLIT_FILE,
    low_memory=False
)

split_df.columns = (
    split_df.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_split_cols = [
    ID_COL,
    "TARGET",
    "SPLIT"
]

missing = [
    c
    for c in required_split_cols
    if c not in split_df.columns
]

if missing:

    raise ValueError(
        f"Golden split thiếu:\n{missing}"
    )

split_df[ID_COL] = (
    standardize_customer_id(
        split_df[ID_COL]
    )
)

split_df["TARGET"] = pd.to_numeric(
    split_df["TARGET"],
    errors="coerce"
).astype(int)

split_df["SPLIT"] = (
    split_df["SPLIT"]
    .astype("string")
    .str.upper()
    .str.strip()
)


# ============================================================
# 13. LOAD GOLDEN E2 CV
# ============================================================

print("\n" + "=" * 110)
print("LOAD GOLDEN E2 CV")
print("=" * 110)

if not E2_CV_FILE.is_file():

    raise FileNotFoundError(
        f"Không tìm thấy:\n{E2_CV_FILE}"
    )

cv_df = pd.read_csv(
    E2_CV_FILE,
    low_memory=False
)

cv_df.columns = (
    cv_df.columns
    .astype(str)
    .str.strip()
    .str.upper()
)

required_cv_cols = [
    ID_COL,
    "TARGET",
    "CV_FOLD"
]

missing = [
    c
    for c in required_cv_cols
    if c not in cv_df.columns
]

if missing:

    raise ValueError(
        f"Golden CV thiếu:\n{missing}"
    )

cv_df[ID_COL] = (
    standardize_customer_id(
        cv_df[ID_COL]
    )
)

cv_df["TARGET"] = pd.to_numeric(
    cv_df["TARGET"],
    errors="coerce"
).astype(int)

cv_df["CV_FOLD"] = pd.to_numeric(
    cv_df["CV_FOLD"],
    errors="coerce"
).astype(int)


# ============================================================
# 14. POPULATION CHECK
# ============================================================

ids_a = set(
    df_a[ID_COL]
)

ids_b = set(
    df_b[ID_COL]
)

ids_c = set(
    df_c[ID_COL]
)

ids_split = set(
    split_df[ID_COL]
)

ids_cv = set(
    cv_df[ID_COL]
)

print("\n" + "=" * 110)
print("POPULATION CHECK")
print("=" * 110)

print(
    "A:",
    f"{len(ids_a):,}"
)

print(
    "B:",
    f"{len(ids_b):,}"
)

print(
    "C:",
    f"{len(ids_c):,}"
)

print(
    "E2 split:",
    f"{len(ids_split):,}"
)

print(
    "E2 CV:",
    f"{len(ids_cv):,}"
)

if not (
    ids_a
    == ids_b
    == ids_c
    == ids_split
):

    raise RuntimeError(
        "A/B/C/E2 split population không giống nhau."
    )


# ============================================================
# 15. TARGET CONSISTENCY
# ============================================================

target_a = (
    df_a[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_A"
        }
    )
)

target_b = (
    df_b[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_B"
        }
    )
)

target_c = (
    df_c[
        [ID_COL, "TARGET_SHARED"]
    ]
    .rename(
        columns={
            "TARGET_SHARED": "TARGET_C"
        }
    )
)

targets = (
    target_a
    .merge(
        target_b,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        target_c,
        on=ID_COL,
        validate="one_to_one"
    )
    .merge(
        split_df[
            [ID_COL, "TARGET"]
        ],
        on=ID_COL,
        validate="one_to_one"
    )
)

target_mismatch = (
    (targets["TARGET_A"] != targets["TARGET_B"])
    |
    (targets["TARGET_A"] != targets["TARGET_C"])
    |
    (targets["TARGET_A"] != targets["TARGET"])
)

print(
    "Target mismatch:",
    int(target_mismatch.sum())
)

if target_mismatch.any():

    raise RuntimeError(
        "Target A/B/C/E2 không giống nhau."
    )


# ============================================================
# 16. SHARED TRAIN / TEST
# ============================================================

train_ids = set(
    split_df.loc[
        split_df["SPLIT"] == "TRAIN",
        ID_COL
    ]
)

test_ids = set(
    split_df.loc[
        split_df["SPLIT"] == "TEST",
        ID_COL
    ]
)

print("\n" + "=" * 110)
print("GOLDEN E2 TRAIN / TEST")
print("=" * 110)

print(
    "Train:",
    f"{len(train_ids):,}"
)

print(
    "Test:",
    f"{len(test_ids):,}"
)

if train_ids & test_ids:

    raise RuntimeError(
        "TRAIN/TEST overlap."
    )

if (
    len(train_ids)
    +
    len(test_ids)
    !=
    len(ids_split)
):

    raise RuntimeError(
        "TRAIN + TEST không bằng population."
    )


# ============================================================
# 17. SHARED CV
# ============================================================

train_cv = cv_df[
    cv_df[ID_COL].isin(
        train_ids
    )
].copy()

if len(train_cv) != len(train_ids):

    raise RuntimeError(
        "Golden CV không chứa đủ TRAIN customers."
    )

print("\n" + "=" * 110)
print("GOLDEN E2 CV")
print("=" * 110)

print(
    train_cv["CV_FOLD"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 18. BUILD PREPROCESSOR
# ============================================================

def build_preprocessor(
    X_train
):

    numeric_cols = (
        X_train
        .select_dtypes(
            include=np.number
        )
        .columns
        .tolist()
    )

    categorical_cols = (
        X_train
        .select_dtypes(
            exclude=np.number
        )
        .columns
        .tolist()
    )

    numeric_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=0
            )
        )
    ])

    categorical_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="MISSING"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    preprocessor = ColumnTransformer([
        (
            "num",
            numeric_pipe,
            numeric_cols
        ),
        (
            "cat",
            categorical_pipe,
            categorical_cols
        )
    ])

    return preprocessor


# ============================================================
# 19. METRICS
# ============================================================

def ks_score(
    y_true,
    probability
):

    fpr, tpr, _ = roc_curve(
        y_true,
        probability
    )

    return float(
        np.max(
            tpr - fpr
        )
    )


def bss_score(
    y_true,
    probability
):

    brier = brier_score_loss(
        y_true,
        probability
    )

    prevalence = float(
        np.mean(
            y_true
        )
    )

    reference = (
        prevalence
        *
        (1 - prevalence)
    )

    if reference <= 0:
        return np.nan

    return float(
        1
        -
        brier / reference
    )


def ranking_metrics(
    y_true,
    probability
):

    temp = pd.DataFrame({

        "Y_TRUE":
            np.asarray(
                y_true
            ),

        "PROB":
            np.asarray(
                probability
            )
    })

    temp = (
        temp
        .sort_values(
            "PROB",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )

    base_rate = float(
        temp["Y_TRUE"].mean()
    )

    total_positive = int(
        temp["Y_TRUE"].sum()
    )

    rows = []

    for rate in [
        0.10,
        0.20,
        0.30
    ]:

        n_top = max(
            1,
            int(
                np.ceil(
                    len(temp)
                    *
                    rate
                )
            )
        )

        top_df = temp.iloc[
            :n_top
        ]

        precision = float(
            top_df["Y_TRUE"].mean()
        )

        capture = (
            float(
                top_df["Y_TRUE"].sum()
            )
            /
            total_positive
            if total_positive > 0
            else np.nan
        )

        lift = (
            precision
            /
            base_rate
            if base_rate > 0
            else np.nan
        )

        rows.append({

            "TOP_RATE":
                rate,

            "PRECISION":
                precision,

            "LIFT":
                lift,

            "CAPTURE":
                capture
        })

    return pd.DataFrame(
        rows
    )


# ============================================================
# 20. PREPARE BRANCH
# ============================================================

def prepare_branch(
    df,
    feature_cols
):

    missing = [
        c
        for c in feature_cols
        if c not in df.columns
    ]

    if missing:

        raise ValueError(
            f"Missing features:\n{missing}"
        )

    data = df[
        [ID_COL, "TARGET_SHARED"]
        + feature_cols
    ].copy()

    data = (
        data
        .sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )

    return data


data_a = prepare_branch(
    df_a,
    FEATURES_A
)

data_b = prepare_branch(
    df_b,
    FEATURES_B
)

data_c = prepare_branch(
    df_c,
    FEATURES_C
)


# ============================================================
# 21. RUN ONE CONTROLLED BRANCH
# ============================================================

def run_branch(
    branch_name,
    data,
    feature_cols
):

    print("\n" + "=" * 110)
    print(f"RUN {branch_name}")
    print("=" * 110)

    # --------------------------------------------------------
    # Verify exact same population
    # --------------------------------------------------------

    branch_ids = set(
        data[ID_COL]
    )

    if branch_ids != ids_split:

        raise RuntimeError(
            f"{branch_name}: "
            "customer population không khớp E2."
        )

    # --------------------------------------------------------
    # Use EXACT E2 train/test IDs
    # --------------------------------------------------------

    train_mask = (
        data[ID_COL]
        .isin(train_ids)
    )

    test_mask = (
        data[ID_COL]
        .isin(test_ids)
    )

    X = data[
        feature_cols
    ].copy()

    y = (
        data["TARGET_SHARED"]
        .astype(int)
    )

    X_train = (
        X.loc[
            train_mask
        ]
        .reset_index(
            drop=True
        )
    )

    X_test = (
        X.loc[
            test_mask
        ]
        .reset_index(
            drop=True
        )
    )

    y_train = (
        y.loc[
            train_mask
        ]
        .reset_index(
            drop=True
        )
    )

    y_test = (
        y.loc[
            test_mask
        ]
        .reset_index(
            drop=True
        )
    )

    train_customer_ids = (
        data.loc[
            train_mask,
            ID_COL
        ]
        .reset_index(
            drop=True
        )
    )

    test_customer_ids = (
        data.loc[
            test_mask,
            ID_COL
        ]
        .reset_index(
            drop=True
        )
    )

    print(
        "Train:",
        f"{len(X_train):,}"
    )

    print(
        "Test:",
        f"{len(X_test):,}"
    )

    print(
        "Features:",
        len(feature_cols)
    )

    # --------------------------------------------------------
    # FOLD RESULTS
    # --------------------------------------------------------

    cv_rows = []

    oof_probability = np.full(
        len(X_train),
        np.nan
    )

    # --------------------------------------------------------
    # EXACT GOLDEN FOLDS
    # --------------------------------------------------------

    for fold in range(
        1,
        N_SPLITS + 1
    ):

        valid_ids = set(
            train_cv.loc[
                train_cv["CV_FOLD"] == fold,
                ID_COL
            ]
        )

        fold_train_ids = (
            train_ids
            -
            valid_ids
        )

        fold_train_mask = (
            train_customer_ids
            .isin(
                fold_train_ids
            )
            .to_numpy()
        )

        fold_valid_mask = (
            train_customer_ids
            .isin(
                valid_ids
            )
            .to_numpy()
        )

        X_fold_train = (
            X_train.loc[
                fold_train_mask
            ]
            .copy()
        )

        X_fold_valid = (
            X_train.loc[
                fold_valid_mask
            ]
            .copy()
        )

        y_fold_train = (
            y_train[
                fold_train_mask
            ]
        )

        y_fold_valid = (
            y_train[
                fold_valid_mask
            ]
        )

        # ----------------------------------------------------
        # PREPROCESS ONLY ON FOLD TRAIN
        # ----------------------------------------------------

        preprocessor = build_preprocessor(
            X_fold_train
        )

        X_fold_train_processed = (
            preprocessor
            .fit_transform(
                X_fold_train
            )
        )

        X_fold_valid_processed = (
            preprocessor
            .transform(
                X_fold_valid
            )
        )

        # ----------------------------------------------------
        # XGBOOST — NO SAMPLING
        # ----------------------------------------------------

        model = get_xgb()

        model.fit(
            X_fold_train_processed,
            y_fold_train
        )

        fold_probability = (
            model
            .predict_proba(
                X_fold_valid_processed
            )[:, 1]
        )

        # ----------------------------------------------------
        # OOF POSITION
        # ----------------------------------------------------

        valid_positions = np.where(
            fold_valid_mask
        )[0]

        oof_probability[
            valid_positions
        ] = fold_probability

        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        fold_auc = roc_auc_score(
            y_fold_valid,
            fold_probability
        )

        fold_ks = ks_score(
            y_fold_valid,
            fold_probability
        )

        fold_brier = brier_score_loss(
            y_fold_valid,
            fold_probability
        )

        fold_bss = bss_score(
            y_fold_valid,
            fold_probability
        )

        cv_rows.append({

            "FOLD":
                fold,

            "AUC":
                fold_auc,

            "KS":
                fold_ks,

            "BRIER":
                fold_brier,

            "BSS":
                fold_bss
        })

        print(
            f"Fold {fold}: "
            f"AUC={fold_auc:.4f} | "
            f"KS={fold_ks:.4f} | "
            f"Brier={fold_brier:.4f}"
        )

        del model
        del preprocessor
        del X_fold_train_processed
        del X_fold_valid_processed

        gc.collect()

    cv_results = pd.DataFrame(
        cv_rows
    )

    # --------------------------------------------------------
    # FINAL MODEL ON EXACT 80% TRAIN
    # --------------------------------------------------------

    final_preprocessor = build_preprocessor(
        X_train
    )

    X_train_processed = (
        final_preprocessor
        .fit_transform(
            X_train
        )
    )

    X_test_processed = (
        final_preprocessor
        .transform(
            X_test
        )
    )

    final_model = get_xgb()

    final_model.fit(
        X_train_processed,
        y_train
    )

    test_probability = (
        final_model
        .predict_proba(
            X_test_processed
        )[:, 1]
    )

    # --------------------------------------------------------
    # TEST METRICS
    # --------------------------------------------------------

    test_auc = roc_auc_score(
        y_test,
        test_probability
    )

    test_ks = ks_score(
        y_test,
        test_probability
    )

    test_brier = brier_score_loss(
        y_test,
        test_probability
    )

    test_bss = bss_score(
        y_test,
        test_probability
    )

    ranking = ranking_metrics(
        y_test,
        test_probability
    )

    # --------------------------------------------------------
    # STORE TEST PREDICTIONS
    # --------------------------------------------------------

    predictions = pd.DataFrame({

        ID_COL:
            test_customer_ids
            .to_numpy(),

        "Y_TRUE":
            y_test
            .to_numpy(),

        "Y_PROB":
            test_probability
    })

    predictions = (
        predictions
        .sort_values(
            ID_COL
        )
        .reset_index(
            drop=True
        )
    )

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    result = {

        "Branch":
            branch_name,

        "Features":
            len(feature_cols),

        "CV_AUC":
            float(
                cv_results["AUC"].mean()
            ),

        "CV_AUC_SD":
            float(
                cv_results["AUC"].std(
                    ddof=1
                )
            ),

        "TEST_AUC":
            float(
                test_auc
            ),

        "KS":
            float(
                test_ks
            ),

        "BSS":
            float(
                test_bss
            ),

        "BRIER":
            float(
                test_brier
            ),

        "Precision@10":
            float(
                ranking.iloc[0]["PRECISION"]
            ),

        "Lift@10":
            float(
                ranking.iloc[0]["LIFT"]
            ),

        "Capture@10":
            float(
                ranking.iloc[0]["CAPTURE"]
            ),

        "Precision@20":
            float(
                ranking.iloc[1]["PRECISION"]
            ),

        "Lift@20":
            float(
                ranking.iloc[1]["LIFT"]
            ),

        "Capture@20":
            float(
                ranking.iloc[1]["CAPTURE"]
            ),

        "Precision@30":
            float(
                ranking.iloc[2]["PRECISION"]
            ),

        "Lift@30":
            float(
                ranking.iloc[2]["LIFT"]
            ),

        "Capture@30":
            float(
                ranking.iloc[2]["CAPTURE"]
            )
    }

    print("\nFINAL TEST RESULT")

    print(
        f"AUC        = {test_auc:.6f}"
    )

    print(
        f"KS         = {test_ks:.6f}"
    )

    print(
        f"Brier      = {test_brier:.6f}"
    )

    print(
        f"BSS        = {test_bss:.6f}"
    )

    print(
        f"Precision@10 = "
        f"{result['Precision@10']:.6f}"
    )

    print(
        f"Lift@10      = "
        f"{result['Lift@10']:.6f}"
    )

    print(
        f"Capture@10   = "
        f"{result['Capture@10']:.6f}"
    )

    return {
        "summary":
            result,

        "cv":
            cv_results,

        "ranking":
            ranking,

        "predictions":
            predictions,

        "model":
            final_model,

        "preprocessor":
            final_preprocessor
    }


# ============================================================
# 22. RUN A / B / C
# ============================================================

result_a = run_branch(
    "A — Fixed first 60 days",
    data_a,
    FEATURES_A
)

gc.collect()

result_b = run_branch(
    "B — Event anchored",
    data_b,
    FEATURES_B
)

gc.collect()

result_c = run_branch(
    "C — Event anchored + tenure",
    data_c,
    FEATURES_C
)

gc.collect()


# ============================================================
# 23. EXTRACT PREDICTIONS
# ============================================================

pred_a = (
    result_a["predictions"]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

pred_b = (
    result_b["predictions"]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)

pred_c = (
    result_c["predictions"]
    .sort_values(
        ID_COL
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 24. VERIFY SAME TEST CUSTOMERS
# ============================================================

test_set_a = set(
    pred_a[ID_COL]
)

test_set_b = set(
    pred_b[ID_COL]
)

test_set_c = set(
    pred_c[ID_COL]
)

print(
    "\n" + "=" * 110
)

print(
    "TEST CUSTOMER VERIFICATION"
)

print(
    "=" * 110
)

print(
    "A == B:",
    test_set_a == test_set_b
)

print(
    "B == C:",
    test_set_b == test_set_c
)

print(
    "A == C:",
    test_set_a == test_set_c
)

if not (
    test_set_a
    ==
    test_set_b
    ==
    test_set_c
):

    raise RuntimeError(
        "A/B/C không cùng test customers."
    )


# ============================================================
# 25. PREPARE TABLE 10
# ============================================================

summary_df = pd.DataFrame([

    result_a["summary"],

    result_b["summary"],

    result_c["summary"]

])


# ------------------------------------------------------------
# AUC DELTAS
# ------------------------------------------------------------

auc_a = (
    result_a["summary"]["TEST_AUC"]
)

auc_b = (
    result_b["summary"]["TEST_AUC"]
)

auc_c = (
    result_c["summary"]["TEST_AUC"]
)

delta_window = (
    auc_b
    -
    auc_a
)

delta_explicit = (
    auc_c
    -
    auc_b
)


# ============================================================
# 26. TABLE 10 — EXACT PAPER FORMAT
# ============================================================

table10 = pd.DataFrame({

    "Br.": [
        "A",
        "B",
        "C"
    ],

    "Feature construction": [

        "Fixed first 60 days",

        "Event anchored",

        "Event anchored + tenure"
    ],

    "AUC": [

        auc_a,

        auc_b,

        auc_c
    ],

    "ΔAUC": [

        np.nan,

        delta_window,

        delta_explicit
    ],

    "KS": [

        result_a["summary"]["KS"],

        result_b["summary"]["KS"],

        result_c["summary"]["KS"]
    ],

    "Lift@10%": [

        result_a["summary"]["Lift@10"],

        result_b["summary"]["Lift@10"],

        result_c["summary"]["Lift@10"]
    ],

    "C@10%": [

        result_a["summary"]["Capture@10"],

        result_b["summary"]["Capture@10"],

        result_c["summary"]["Capture@10"]
    ],

    "BSS": [

        result_a["summary"]["BSS"],

        result_b["summary"]["BSS"],

        result_c["summary"]["BSS"]
    ]
})


# ============================================================
# 27. PAIRWISE SUMMARY
# ============================================================

pairwise = pd.DataFrame({

    "Comparison": [

        "B - A",

        "C - B",

        "C - A"
    ],

    "AUC_1": [

        auc_a,

        auc_b,

        auc_a
    ],

    "AUC_2": [

        auc_b,

        auc_c,

        auc_c
    ],

    "Delta_AUC": [

        auc_b - auc_a,

        auc_c - auc_b,

        auc_c - auc_a
    ]
})


# ============================================================
# 28. PRINT TABLE 10
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TABLE 10 — SAME-POPULATION DECOMPOSITION"
)

print(
    "=" * 110
)

print(
    table10.to_string(
        index=False,
        float_format=lambda x:
            f"{x:.4f}"
    )
)


print(
    "\nDwindow = "
    f"{delta_window:.6f}"
)

print(
    "Dexplicit = "
    f"{delta_explicit:.6f}"
)

print(
    "C - A = "
    f"{auc_c - auc_a:.6f}"
)


# ============================================================
# 29. SAVE CSV
# ============================================================

table10.to_csv(
    TABLE10_FILE,
    index=False,
    encoding="utf-8-sig"
)

result_a["cv"].to_csv(
    CV_A_FILE,
    index=False,
    encoding="utf-8-sig"
)

result_b["cv"].to_csv(
    CV_B_FILE,
    index=False,
    encoding="utf-8-sig"
)

result_c["cv"].to_csv(
    CV_C_FILE,
    index=False,
    encoding="utf-8-sig"
)

pred_a.to_csv(
    PRED_A_FILE,
    index=False,
    encoding="utf-8-sig"
)

pred_b.to_csv(
    PRED_B_FILE,
    index=False,
    encoding="utf-8-sig"
)

pred_c.to_csv(
    PRED_C_FILE,
    index=False,
    encoding="utf-8-sig"
)

pairwise.to_csv(
    PAIRWISE_FILE,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 30. EXCEL
# ============================================================

with pd.ExcelWriter(
    EXCEL_FILE,
    engine="openpyxl"
) as writer:

    table10.to_excel(
        writer,
        sheet_name="Table 10",
        index=False
    )

    summary_df.to_excel(
        writer,
        sheet_name="Full Summary",
        index=False
    )

    pairwise.to_excel(
        writer,
        sheet_name="Pairwise Delta AUC",
        index=False
    )

    result_a["cv"].to_excel(
        writer,
        sheet_name="A CV",
        index=False
    )

    result_b["cv"].to_excel(
        writer,
        sheet_name="B CV",
        index=False
    )

    result_c["cv"].to_excel(
        writer,
        sheet_name="C CV",
        index=False
    )

    pred_a.to_excel(
        writer,
        sheet_name="A Predictions",
        index=False
    )

    pred_b.to_excel(
        writer,
        sheet_name="B Predictions",
        index=False
    )

    pred_c.to_excel(
        writer,
        sheet_name="C Predictions",
        index=False
    )

    split_df.to_excel(
        writer,
        sheet_name="Golden E2 Split",
        index=False
    )

    cv_df.to_excel(
        writer,
        sheet_name="Golden E2 CV",
        index=False
    )


# ============================================================
# 31. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 110
)

print(
    "TABLE 10 COMPLETED"
)

print(
    "=" * 110
)

print(
    "\nPopulation:",
    f"{len(ids_split):,}"
)

print(
    "Train:",
    f"{len(train_ids):,}"
)

print(
    "Test:",
    f"{len(test_ids):,}"
)

print(
    "\nA AUC:",
    f"{auc_a:.6f}"
)

print(
    "B AUC:",
    f"{auc_b:.6f}"
)

print(
    "C AUC:",
    f"{auc_c:.6f}"
)

print(
    "\nDwindow = B - A:",
    f"{delta_window:.6f}"
)

print(
    "Dexplicit = C - B:",
    f"{delta_explicit:.6f}"
)

print(
    "\nSaved Table 10:"
)

print(
    TABLE10_FILE
)

print(
    "\nSaved Excel:"
)

print(
    EXCEL_FILE
)

MOUNT GOOGLE DRIVE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

LOAD A / B / C

--------------------------------------------------------------------------------------------------------------
LOAD A
/content/drive/MyDrive/Fintect/final_landmark_60d/final_dataset_landmark_60d_no_auto_job.csv
--------------------------------------------------------------------------------------------------------------
Shape: (94453, 21)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD B
/content/drive/MyDrive/Fintect/VIB_FLDC_60D.csv
--------------------------------------------------------------------------------------------------------------
Shape: (94453, 25)
Customers: 94,453
Positive: 4374

--------------------------------------------------------------------------------------------------------------
LOAD C
/content/dri